In [ ]:
def main(datasources, start_date, end_date):
    """
    构建四因子截面标准化等权复合因子。

    四个成分因子分别对应相关性去重后的第 2、4、9、17 名。
    每日先在中证 1000 截面内做 z-score，再按 25% 等权合成。
    """
    import numpy as np
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]
    output_start = pd.to_datetime(start_date).normalize()
    output_end = pd.to_datetime(end_date).normalize()
    query_start = output_start.strftime("%Y-%m-%d 00:00:00")
    query_end = (
        output_end + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    ).strftime("%Y-%m-%d %H:%M:%S")

    sql = f"""
    WITH base AS (
        SELECT
            date,
            date::DATE::DATETIME AS trading_day,
            instrument,
            open,
            close,
            high,
            deal_number,
            (bid_num_orders1 - ask_num_orders1) * 1.0
                / NULLIF(bid_num_orders1 + ask_num_orders1, 0)
                AS order_imbalance1
        FROM {bar1m}
    ),
    minute_window AS (
        SELECT
            *,
            LAG(high, 1) OVER intraday_window AS previous_high,
            LAG(order_imbalance1, 1) OVER intraday_window
                AS previous_order_imbalance1
        FROM base
        WINDOW intraday_window AS (
            PARTITION BY trading_day, instrument
            ORDER BY date
        )
    ),
    daily AS (
        SELECT
            trading_day AS date,
            instrument,
            LAST(open / close ORDER BY date) AS open_close_last,
            LAST(deal_number ORDER BY date) AS deal_number_last,
            SUM(high / NULLIF(previous_high, 0) - 1)
                AS high_return_sum,
            nanstd(
                order_imbalance1
                    / NULLIF(previous_order_imbalance1, 0) - 1
            ) AS order_imbalance_return_std
        FROM minute_window
        GROUP BY trading_day, instrument
    )
    SELECT
        date,
        instrument,
        open_close_last,
        deal_number_last,
        high_return_sum,
        order_imbalance_return_std
    FROM daily
    """

    daily = dai.query(
        sql,
        filters={"date": [query_start, query_end]},
        compression=True,
    ).df()
    daily["date"] = pd.to_datetime(daily["date"])
    daily["instrument"] = daily["instrument"].astype(str)

    stock_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [query_start, query_end]},
    ).df()
    stock_pool["date"] = pd.to_datetime(stock_pool["date"])
    stock_pool["instrument"] = stock_pool["instrument"].astype(str)
    daily = pd.merge(
        daily, stock_pool, how="inner", on=["date", "instrument"]
    )

    # 还原四个已经统一为正向的 GP 因子。
    daily["factor_02"] = daily.groupby("date")[
        "open_close_last"
    ].rank(pct=True)
    daily["factor_04"] = -(daily["deal_number_last"] - 1.0) * 2.0
    daily["factor_09"] = -daily["high_return_sum"].abs() * 2.0
    daily["factor_17"] = daily.groupby("date")[
        "order_imbalance_return_std"
    ].rank(pct=True)

    factor_columns = ["factor_02", "factor_04", "factor_09", "factor_17"]
    standardized_columns = []
    for column in factor_columns:
        values = pd.to_numeric(daily[column], errors="coerce").replace(
            [np.inf, -np.inf], np.nan
        )
        daily_mean = values.groupby(daily["date"]).transform("mean")
        daily_std = values.groupby(daily["date"]).transform("std")
        standardized = f"{column}_zscore"
        daily[standardized] = (values - daily_mean) / daily_std
        standardized_columns.append(standardized)

    daily["factor"] = (
        daily[standardized_columns].fillna(0.0).sum(axis=1) / 4.0
    )
    result = daily.loc[
        (daily["date"] >= output_start) & (daily["date"] <= output_end),
        ["date", "instrument", "factor"],
    ].copy()
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce")
    result["factor"] = result["factor"].replace(
        [np.inf, -np.inf], np.nan
    )
    result = result.dropna(subset=["factor"])
    return result.sort_values(["date", "instrument"]).reset_index(drop=True)
